# 🎨 AUTO-SCRIBE V2: TRUNG TÂM ĐIỀU KHIỂN & CẦU NỐI AI AGENT LAPTOP
Quy trình 3 bước chuyên nghiệp:
- 🤖 **Bước 1**: Gửi câu thoại về Laptop ➔ Antigravity (AI IDE) phân tích kịch bản (0đ API Key).
- 🖼️ **Bước 2**: Kéo kịch bản về Colab ➔ Tải/Vẽ ảnh SVG nét viền (Hollow Outline Doodle, không tô đen) ➔ **Hiển thị Bảng Xem Trước (Chưa xuất file vội)**.
- 🎨 **Chỉnh sửa**: Xem ảnh trực quan, bấm "Vẽ Lại" nếu chưa ưng ý.
- 📦 **Bước 3**: Khi đã hài lòng ➔ Bấm **Đóng Gói & Tải File VideoScribe** (Có đầy đủ âm thanh voiceOver.mp3).

## ⚡ BƯỚC 1: Cài Đặt Môi Trường & Kết Nối Google Drive (Chạy 1 lần)

In [ ]:
from google.colab import drive
import os
import sys

print("🔗 Đang yêu cầu quyền truy cập Google Drive...")
drive.mount('/content/drive')

print("⏳ Đang cài đặt thư viện lõi (Whisper, Gemini, vtracer, Gradio, Pillow, ffmpeg)...")
!apt-get install -y ffmpeg
!pip install -q openai-whisper google-genai requests vtracer Pillow gradio

print("✅ Đã cài đặt xong toàn bộ môi trường! Hãy chuyển sang BƯỚC 2 để mở Giao Diện.")

## 🎛️ BƯỚC 2: Khởi Chạy Giao Diện Web Điều Khiển Toàn Diện (All-In-One UI)

In [ ]:
import os
import re
import json
import time
import random
import shutil
import zipfile
import subprocess
import urllib.request
import urllib.parse
from PIL import Image
import vtracer
import whisper
import requests
import gradio as gr

ASSETS_DIR = "assets"
os.makedirs(ASSETS_DIR, exist_ok=True)

def clean_slug(text):
    text = re.sub(r'[^a-zA-Z0-9\s_-]', '', text)
    text = re.sub(r'\s+', '_', text).strip('_').lower()
    return text[:40] if text else "doodle_icon"

def videoscribe_escape(s):
    s = s.replace('&', '&amp;')
    s = s.replace('<', '&lt;')
    s = s.replace('"', '&quot;')
    return s

def ensure_svg_file(input_file_path, output_svg_path):
    """Đảm bảo file luôn là 1 file SVG text hợp lệ, tự động vector hóa nếu là PNG/JPG."""
    os.makedirs(os.path.dirname(os.path.abspath(output_svg_path)), exist_ok=True)
    
    is_bitmap = False
    if os.path.exists(input_file_path):
        try:
            with open(input_file_path, 'rb') as check_f:
                header = check_f.read(8)
                if header.startswith(b'\x89PNG') or header.startswith(b'\xff\xd8') or header.startswith(b'GIF8') or header.startswith(b'RIFF'):
                    is_bitmap = True
        except Exception: pass

    if is_bitmap or input_file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        temp_png = output_svg_path + ".temp_norm.png"
        try:
            with Image.open(input_file_path) as img:
                img = img.convert('RGB')
                img.save(temp_png, 'PNG')
                
            vtracer.convert_image_to_svg_py(
                temp_png,
                output_svg_path,
                colormode='binary',
                hierarchical='stacked',
                mode='spline',
                filter_speckle=4,
                corner_threshold=60,
                length_threshold=4.0,
                max_iterations=10,
                splice_threshold=45,
                path_precision=3
            )
            if os.path.exists(temp_png): os.remove(temp_png)
            return output_svg_path
        except Exception:
            if os.path.exists(temp_png): os.remove(temp_png)
            
    if os.path.exists(input_file_path):
        if os.path.abspath(input_file_path) != os.path.abspath(output_svg_path):
            shutil.copy(input_file_path, output_svg_path)
    else:
        with open(output_svg_path, "w", encoding="utf-8") as f:
            f.write('<svg xmlns="http://www.w3.org/2000/svg" width="500" height="500"><rect width="500" height="500" fill="none" stroke="#000" stroke-width="4"/><text x="250" y="250" font-size="26" text-anchor="middle" fill="#000">Doodle</text></svg>')
    return output_svg_path

def build_drive_cache(drive_search_dirs):
    cache = []
    for s_dir in drive_search_dirs:
        if os.path.exists(s_dir):
            for root, dirs, files in os.walk(s_dir):
                for f in files:
                    if f.lower().endswith('.svg') or f.lower().endswith('.png') or f.lower().endswith('.jpg'):
                        clean_n = re.sub(r'[^a-zA-Z0-9]', ' ', os.path.splitext(f)[0]).lower()
                        cache.append({
                            "path": os.path.join(root, f),
                            "filename": f,
                            "words": set(clean_n.split()),
                            "clean_name": clean_n
                        })
    return cache

def generate_doodle_svg(prompt_keyword, target_svg_path, drive_save_path=None):
    os.makedirs(os.path.dirname(os.path.abspath(target_svg_path)), exist_ok=True)
    ai_prompt = (
        f"simple black outline whiteboard doodle sketch of {prompt_keyword}, fine line art, pure white background, "
        f"hollow transparent outlines, clean thin black pen strokes, coloring book line drawing style, no black fill, no solid black shapes, no shading, minimalist doodle outline"
    )
    encoded_prompt = urllib.parse.quote(ai_prompt)
    seed = random.randint(1000, 999999)
    url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=768&height=768&model=flux&nologo=true&seed={seed}"
    
    temp_img = target_svg_path + ".temp_dl"
    temp_png = target_svg_path + ".temp.png"
    success = False
    
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=25) as resp:
                with open(temp_img, 'wb') as f:
                    f.write(resp.read())
            
            with Image.open(temp_img) as img:
                img = img.convert('RGB')
                img.save(temp_png, 'PNG')
            
            vtracer.convert_image_to_svg_py(
                temp_png,
                target_svg_path,
                colormode='binary',
                hierarchical='stacked',
                mode='spline',
                filter_speckle=6,
                corner_threshold=70,
                length_threshold=5.0,
                max_iterations=10,
                splice_threshold=50,
                path_precision=3
            )
            success = True
            break
        except Exception:
            time.sleep(1.5)
        finally:
            if os.path.exists(temp_img): os.remove(temp_img)
            if os.path.exists(temp_png): os.remove(temp_png)
            
    if success and os.path.exists(target_svg_path) and os.path.getsize(target_svg_path) > 100:
        if drive_save_path:
            try:
                os.makedirs(os.path.dirname(os.path.abspath(drive_save_path)), exist_ok=True)
                if os.path.abspath(target_svg_path) != os.path.abspath(drive_save_path):
                    shutil.copy(target_svg_path, drive_save_path)
            except Exception: pass
        return target_svg_path
    else:
        with open(target_svg_path, "w", encoding="utf-8") as f:
            f.write(f'<svg xmlns="http://www.w3.org/2000/svg" width="500" height="500"><rect width="500" height="500" fill="none" stroke="#000" stroke-width="4"/><text x="250" y="250" font-size="26" text-anchor="middle" fill="#000">{prompt_keyword}</text></svg>')
        return target_svg_path

def search_or_generate_svg_fast(query, drive_cache, drive_gen_dir, target_asset_path, used_files=None):
    if used_files is None: used_files = set()
    stop_words = {"vector", "illustration", "clipart", "transparent", "icon", "svg", "drawing", "the", "a", "an"}
    raw_words = re.sub(r'[^a-zA-Z0-9]', ' ', query).lower().split()
    query_words = set([w for w in raw_words if w not in stop_words and len(w) > 1])
    
    best_matches = []
    max_score = 0
    
    for item in drive_cache:
        score = len(query_words.intersection(item["words"]))
        if score > 0:
            if " ".join(query_words) in item["clean_name"]: score += 2.0
            if score > max_score:
                max_score = score
                best_matches = [item["path"]]
            elif score == max_score:
                best_matches.append(item["path"])
                
    if best_matches and max_score >= 1.0:
        unused = [m for m in best_matches if m not in used_files]
        chosen = random.choice(unused) if unused else random.choice(best_matches)
        used_files.add(chosen)
        ensure_svg_file(chosen, target_asset_path)
        return target_asset_path, f"Drive: {os.path.basename(chosen)}"
    
    slug_n = clean_slug(query)
    drive_save = os.path.join(drive_gen_dir, f"{slug_n}.svg") if drive_gen_dir else None
    if drive_save and os.path.exists(drive_save):
        drive_save = os.path.join(drive_gen_dir, f"{slug_n}_{random.randint(100,999)}.svg")
        
    generate_doodle_svg(query, target_asset_path, drive_save)
    time.sleep(0.5)
    return target_asset_path, f"AI Mới: {os.path.basename(drive_save) if drive_save else 'Local'}"

def build_scribe_file(audio_path, metadata_path="scene_metadata.json"):
    if not os.path.exists(metadata_path): return None
    with open(metadata_path, "r", encoding="utf-8") as f:
        meta_data = json.load(f)

    BUILD_DIR = "build_scribe"
    if os.path.exists(BUILD_DIR): shutil.rmtree(BUILD_DIR)
    os.makedirs(BUILD_DIR, exist_ok=True)

    # 1. Chuẩn hóa âm thanh voiceover.mp3 cho VideoScribe
    audio_dst = os.path.join(BUILD_DIR, "voiceover.mp3")
    has_audio = False
    if audio_path and os.path.exists(audio_path):
        try:
            subprocess.run(['ffmpeg', '-y', '-i', audio_path, '-ar', '44100', '-ac', '2', '-b:a', '192k', audio_dst], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
            has_audio = True
        except Exception:
            shutil.copy(audio_path, audio_dst)
            has_audio = True

    # 2. Tạo drawing.xml với voiceOver="voiceover.mp3" CHUẨN XÁC ĐỂ PHÁT ÂM THANH
    drawing_xml = os.path.join(BUILD_DIR, "drawing.xml")
    voiceover_attr = 'voiceOver="voiceover.mp3" voiceOverVolume="100"' if has_audio else 'voiceOver="" voiceOverVolume="100"'
    
    with open(drawing_xml, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="utf-8"?>\n')
        f.write(f'<drawing app="VideoScribe" ver="3.7.3103" filever="5" name="Auto_Project" desc="" tags="" uniqueID="{random.randint(100000000, 999999999)}" isDescendedFromTemplate="not_desc" defaultHandMD5="default_right" options="&lt;drawingOptions paperStyle=&quot;1&quot; paperColour=&quot;16777215&quot; threeDMode=&quot;no&quot; loopSound=&quot;no&quot; zoomAtEnd=&quot;no&quot; vignette=&quot;0&quot; xPerspective=&quot;0&quot; yPerspective=&quot;0&quot; zPerspective=&quot;0&quot;/&gt;" backingTrack="" lastRenderDateTime="Invalid Date" {voiceover_attr}>\n')
        
        xml_elements = []
        element_counter = 1000000000 + random.randint(10000, 99999)
        visual_timeline_ms = 0
        
        for scene_idx, scene in enumerate(meta_data):
            raw_images = scene.get('images', [])
            n = len(raw_images)
            if n == 0: continue
            
            speech_dur_s = float(scene.get('end', 0)) - float(scene.get('start', 0))
            if speech_dur_s <= 0: speech_dur_s = 3.5 * n
            duration_per_img = speech_dur_s / n
            scene_x = scene_idx * 1600
            scene_y = 0
            cam_scale = 0.82
            cam_x = 448.5 - scene_x * cam_scale
            cam_y = 252.5 - scene_y * cam_scale
            
            for i, img_meta in enumerate(raw_images):
                filename = img_meta.get('file_name', '')
                file_path = os.path.join(ASSETS_DIR, filename)
                
                if not os.path.exists(file_path):
                    alt_svg = os.path.splitext(file_path)[0] + ".svg"
                    if os.path.exists(alt_svg): file_path = alt_svg
                    else: continue
                
                ensure_svg_file(file_path, file_path)
                actual_filename = os.path.basename(file_path)

                content = ""
                try:
                    with open(file_path, "r", encoding="utf-8", errors="replace") as f2:
                        raw_svg = f2.read()
                        raw_svg = re.sub(r'<\?xml[^>]*\?>', '', raw_svg)
                        raw_svg = re.sub(r'<!DOCTYPE[^>]*>', '', raw_svg)
                        raw_svg = re.sub(r'<!--.*?-->', '', raw_svg, flags=re.DOTALL)
                        content = raw_svg.replace('\n', ' ').replace('\r', '')
                except Exception:
                    content = '<svg xmlns="http://www.w3.org/2000/svg" width="500" height="500"></svg>'
                    
                element_counter += random.randint(1000, 5000)
                if n == 1: pos_x, pos_y, scale_val = scene_x, scene_y, "0.8"
                elif n == 2: pos_x, pos_y, scale_val = scene_x + (-250 if i == 0 else 250), scene_y, "0.55"
                else: pos_x, pos_y, scale_val = scene_x + (-220 if i == 1 else (220 if i == 2 else 0)), scene_y + (120 if i > 0 else -120), "0.45"
                    
                ai_style = img_meta.get('animation_style', 'draw')
                if ai_style == 'draw': draw_style = 'draw_style_normal'
                elif ai_style in ['movein', 'movein_hand', 'movein_nohand']: draw_style = 'draw_style_movein'
                elif ai_style == 'fadein': draw_style = 'draw_style_fadein'
                else: draw_style = 'draw_style_normal'
                
                movin_compass = str(random.randint(1, 8))
                draw_detail = 'yes' if draw_style == 'draw_style_normal' else 'no'
                custom_hand = 'default_nohand' if draw_style == 'draw_style_movein' else ''
                movin_arc = random.choice(['0', '1']) if draw_style == 'draw_style_movein' else '0'
                
                total_time_ms = int(duration_per_img * 1000)
                trans_time_ms = min(500, int(total_time_ms * 0.15))
                pause_time_ms = min(500, int(total_time_ms * 0.10))
                target_time_ms = max(0, total_time_ms - trans_time_ms - pause_time_ms)
                
                drawing_xml_attr = f'drawingXML="{videoscribe_escape(content)}"'
                
                element_xml = (
                    f'  <element elementType="drawing" descName="" elementID="{element_counter}" '
                    f'splitTextField="no" drawingText="" fontName="null" {drawing_xml_attr} '
                    f'customHandMD5="{custom_hand}" colourEffect="0" targetTime="{target_time_ms}" '
                    f'pauseTime="{pause_time_ms}" transitionTime="{trans_time_ms}" '
                    f'drawStyle="{draw_style}" rotation="0" visible="true" '
                    f'currentPosX="{pos_x}" currentPosY="{pos_y}" offsetX="{pos_x}" offsetY="{pos_y}" '
                    f'scalesX="0.8" scalesY="0.8" theScale="0.8" targetHeight="800" '
                    f'movinCompass="{movin_compass}" movinFlow="0" movinArc="{movin_arc}" movinAllowRotate="yes" '
                    f'drawDetail="{draw_detail}" sketchStyle="no" brush="0" brushOptions="0" opacity="1" '
                    f'textColour="-1" textAlign="left" textBackwards="no" rtlLanguage="no" textSpacing="0" '
                    f'flipHoriz="no" flipVert="no" locked="no" calligraphy_angle="45" keepRunning="no" '
                    f'loopOptions="Fit to Time" blendMode="normal" filters="&lt;filters/>" morphFromID="0" '
                    f'morphCamera="no" morphRemoveOld="yes" cameraPositionX="{cam_x}" cameraPositionY="{cam_y}" '
                    f'cameraScale="{cam_scale}" cameraCanvasWid="897.7777777777778" cameraCanvasHei="505" '
                    f'availableRecolours="&lt;availableRecolours/>" recolouringSchemes="&lt;recolouringSchemes/>" '
                    f'skinTone="-1" hairColour="-1" highlightColour="-1" customColour1="-1" customColour2="-1" '
                    f'originalOutlineColour="0" greyscaleContrast="70" />\n'
                )
                xml_elements.append(element_xml)
                visual_timeline_ms += total_time_ms
                
        f.write('\n'.join(xml_elements) + '\n')
        f.write('</drawing>')

    out_file = "Auto_Project.scribe"
    with zipfile.ZipFile(out_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(BUILD_DIR):
            for file in files:
                f_p = os.path.join(root, file)
                zipf.write(f_p, os.path.relpath(f_p, BUILD_DIR))

    tmp_p = out_file + ".tmp"
    with zipfile.ZipFile(out_file, 'r') as zin, zipfile.ZipFile(tmp_p, 'w', zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            data = zin.read(item.filename)
            if item.filename == 'drawing.xml':
                data = data.decode('utf-8', errors="replace").replace('&gt;', '>').encode('utf-8')
            zout.writestr(item, data)
    os.replace(tmp_p, out_file)
    return out_file

def generate_preview_cards(meta_data=None):
    if meta_data is None:
        if not os.path.exists("scene_metadata.json"): return "<p style='padding:20px; color:#718096;'>Chưa có dữ liệu kịch bản.</p>"
        with open("scene_metadata.json", "r", encoding="utf-8") as f:
            meta_data = json.load(f)
            
    cards_html = ""
    for s in meta_data:
        sc_id = s.get('sentence_id', 1)
        start_t = float(s.get('start', 0))
        end_t = float(s.get('end', 0))
        dur_t = end_t - start_t
        speech = s.get('speech_text', s.get('text', ''))
        
        imgs_div = ""
        for img_idx, img in enumerate(s.get('images', [])):
            fp = os.path.join(ASSETS_DIR, img.get('file_name', ''))
            svg_content = ""
            if os.path.exists(fp):
                try:
                    with open(fp, "r", encoding="utf-8", errors="replace") as svg_f:
                        raw_c = svg_f.read()
                        if "<svg" in raw_c:
                            svg_content = re.sub(r'<\?xml[^>]*\?>', '', raw_c)
                        else:
                            svg_content = '<div style="color:#718096; font-size:12px;">[Ảnh Vector]</div>'
                except Exception:
                    svg_content = '<div style="color:#718096; font-size:12px;">[Ảnh]</div>'
                    
            imgs_div += f'''
            <div style="background:#fff; border:1px solid #cbd5e0; border-radius:10px; padding:10px; margin:6px; display:inline-block; vertical-align:top; width:150px; text-align:center; box-shadow:0 2px 4px rgba(0,0,0,0.05);">
                <div style="height:110px; display:flex; align-items:center; justify-content:center; overflow:hidden; background:#f8fafc; border-radius:6px; padding:4px;">
                    {svg_content if svg_content else '<div style="color:#a0aec0;">Chưa có ảnh</div>'}
                </div>
                <div style="font-size:12px; font-weight:bold; color:#1a202c; margin-top:8px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;" title="{img.get('svg_search_prompt','')}">
                    {img.get('svg_search_prompt','')}
                </div>
                <div style="font-size:11px; color:#4a5568; margin-top:2px;">🎭 {img.get('animation_style','draw').upper()}</div>
                <div style="font-size:10px; color:#2b6cb0; margin-top:3px; background:#ebf8ff; padding:2px 4px; border-radius:4px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;">
                    {img.get('source','')}
                </div>
            </div>
            '''

        cards_html += f'''
        <div style="background:#f7fafc; border:1px solid #e2e8f0; border-radius:12px; padding:16px; margin-bottom:16px; box-shadow:0 1px 3px rgba(0,0,0,0.05);">
            <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px; border-bottom:1px solid #e2e8f0; padding-bottom:8px;">
                <span style="font-size:16px; font-weight:bold; color:#2b6cb0;">Cảnh #{sc_id}</span>
                <span style="font-size:13px; background:#edf2f7; color:#4a5568; padding:3px 8px; border-radius:12px; font-weight:600;">⏱️ {start_t:.1f}s ➔ {end_t:.1f}s ({dur_t:.1f}s)</span>
            </div>
            <div style="font-size:14px; color:#2d3748; line-height:1.5; margin-bottom:12px; font-style:italic;">
                🗣️ "{speech}"
            </div>
            <div style="display:flex; flex-wrap:wrap; gap:8px;">
                {imgs_div}
            </div>
        </div>
        '''

    return f'''
    <div style="font-family:-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-height:550px; overflow-y:auto; padding:10px;">
        {cards_html}
    </div>
    '''

# --- BƯỚC 1: BÓC TÁCH WHISPER & GỬI VỀ LAPTOP ---
def step1_whisper_and_send_to_laptop(audio_file_obj, bridge_url_input):
    logs = []
    def fmt_log(msg):
        logs.append(f"[{time.strftime('%H:%M:%S')}] {msg}")
        return "\n".join(logs)

    audio_path = audio_file_obj if isinstance(audio_file_obj, str) else (audio_file_obj.name if audio_file_obj else None)
    if not audio_path and os.path.exists("voiceover.mp3"): audio_path = "voiceover.mp3"
    if not audio_path or not os.path.exists(audio_path):
        yield fmt_log("❌ Lỗi: Vui lòng chọn file âm thanh voiceover.mp3!")
        return

    bridge_url = bridge_url_input.strip().rstrip('/') if bridge_url_input else ""
    if not bridge_url:
        yield fmt_log("❌ Lỗi: Vui lòng dán Local Bridge URL (Cloudflare Tunnel) từ Laptop!")
        return

    yield fmt_log(f"🎙️ Âm thanh: {os.path.basename(audio_path)}")
    yield fmt_log("⏳ Whisper đang bóc tách câu thoại...")
    whisper_model = whisper.load_model("base")
    whisper_res = whisper_model.transcribe(audio_path)
    scenes = [{"sentence_id": idx+1, "start": seg["start"], "end": seg["end"], "text": seg["text"].strip()} for idx, seg in enumerate(whisper_res["segments"]) if seg["text"].strip()]
    yield fmt_log(f"✅ Đã bóc tách {len(scenes)} câu thoại!")

    yield fmt_log(f"🌐 Đang gửi {len(scenes)} câu thoại về Laptop qua {bridge_url}...")
    try:
        resp = requests.post(f"{bridge_url}/", json={"scenes": scenes}, timeout=30)
        if resp.status_code == 200:
            yield fmt_log("═"*60)
            yield fmt_log(f"🎉 ĐÃ GỬI THÀNH CÔNG VỀ LAPTOP (File: pending_scenes.json)!")
            yield fmt_log("👉 BÂY GIỜ BẠN HÃY BẢO AI TRONG IDE LAPTOP: 'Hãy phân tích kịch bản pending_scenes.json'.")
            yield fmt_log("👉 Sau khi AI phân tích xong, bạn bấm '📥 2. Kéo Kịch Bản & Tải/Vẽ Ảnh' bên dưới!")
            yield fmt_log("═"*60)
        else:
            yield fmt_log(f"⚠️ Laptop trả về mã lỗi: {resp.status_code}")
    except Exception as e:
        yield fmt_log(f"❌ Lỗi gửi về Laptop: {e}")

# --- BƯỚC 2: KÉO KỊCH BẢN TỪ LAPTOP VỀ & TẢI/VẼ ẢNH (CHỈ XEM TRƯỚC, CHƯA XUẤT SCRIBE) ---
def step2_pull_from_laptop_and_preview(bridge_url_input, drive_f_path, drive_gen_path):
    logs = []
    def fmt_log(msg):
        logs.append(f"[{time.strftime('%H:%M:%S')}] {msg}")
        return "\n".join(logs)

    bridge_url = bridge_url_input.strip().rstrip('/') if bridge_url_input else ""
    if not bridge_url:
        yield fmt_log("❌ Lỗi: Vui lòng dán Local Bridge URL!"), None
        return

    yield fmt_log(f"📥 Đang kéo kịch bản từ Laptop ({bridge_url}/get_analyzed_scenes)..."), None
    try:
        resp = requests.get(f"{bridge_url}/get_analyzed_scenes", timeout=30)
        if resp.status_code != 200 or resp.json().get("status") != "success":
            yield fmt_log("⚠️ Chưa có kịch bản trên Laptop! Hãy bảo AI trong IDE phân tích file pending_scenes.json trước!"), None
            return
        raw_analyzed_data = resp.json().get("data", [])
        yield fmt_log(f"🎉 Nhận thành công {len(raw_analyzed_data)} cảnh kịch bản từ Laptop!"), None
    except Exception as e:
        yield fmt_log(f"❌ Lỗi kết nối tới Laptop: {e}"), None
        return

    yield fmt_log("📂 Đang quét nhanh kho ảnh trên Google Drive..."), None
    drive_search_dirs = [drive_f_path, drive_gen_path]
    os.makedirs(drive_gen_path, exist_ok=True)
    drive_cache = build_drive_cache(drive_search_dirs)
    yield fmt_log(f"✅ Tìm thấy {len(drive_cache)} ảnh có sẵn trong Drive!"), None

    scene_metadata = []
    used_files = set()
    total_scenes = len(raw_analyzed_data)

    for j, s in enumerate(raw_analyzed_data):
        sc_id = s.get("sentence_id", j + 1)
        raw_imgs = s.get("images", [])
        sentence_entry = {
            "sentence_id": sc_id,
            "start": float(s.get('start', 0)),
            "end": float(s.get('end', 0)),
            "speech_text": s.get('speech_text', s.get('text', '')),
            "images": []
        }
        
        for img_idx, img_info in enumerate(raw_imgs):
            file_base = f"sentence_{sc_id:03d}_img_{img_idx+1:02d}"
            kw = img_info.get("svg_search_prompt", "icon")
            target_p = os.path.join(ASSETS_DIR, f"{file_base}.svg")
            
            act_p, src_note = search_or_generate_svg_fast(kw, drive_cache, drive_gen_path, target_p, used_files)
            
            sentence_entry["images"].append({
                "img_idx": img_idx + 1,
                "visual_concept": img_info.get("visual_concept", ""),
                "svg_search_prompt": kw,
                "animation_style": img_info.get("animation_style", "draw"),
                "file_name": os.path.basename(act_p),
                "source": src_note
            })
            
        scene_metadata.append(sentence_entry)
        yield fmt_log(f"   [Cảnh {sc_id}/{total_scenes}] ➔ {len(raw_imgs)} ảnh ({src_note})"), None
        
    with open("scene_metadata.json", "w", encoding="utf-8") as f:
        json.dump(scene_metadata, f, ensure_ascii=False, indent=2)

    yield fmt_log("═"*60), generate_preview_cards(scene_metadata)
    yield fmt_log("✨ ĐÃ TẢI & VẼ TOÀN BỘ ẢNH XONG! Hãy xem trước ảnh bên dưới. Khi đã ưng ý ➔ Bấm nút '📦 BƯỚC 3: ĐÓNG GÓI & TẢI FILE VIDEOSCRIBE'!"), generate_preview_cards(scene_metadata)

# --- BƯỚC 3: ĐÓNG GÓI VÀ TẢI FILE SCRIBE ---
def step3_build_and_download_scribe(audio_file_obj):
    logs = []
    def fmt_log(msg):
        logs.append(f"[{time.strftime('%H:%M:%S')}] {msg}")
        return "\n".join(logs)

    if not os.path.exists("scene_metadata.json"):
        yield fmt_log("❌ Chưa có dữ liệu kịch bản! Hãy chạy Bước 2 trước."), None
        return

    audio_path = audio_file_obj if isinstance(audio_file_obj, str) else (audio_file_obj.name if audio_file_obj else "voiceover.mp3")
    yield fmt_log("📦 Đang đóng gói dự án Auto_Project.scribe kèm âm thanh voiceover.mp3..."), None
    out_file = build_scribe_file(audio_path, "scene_metadata.json")
    yield fmt_log("🎉 HOÀN TẤT! File dự án VideoScribe đã sẵn sàng để tải về ở khung bên phải."), out_file

# --- VẼ LẠI ẢNH CHO CẢNH ĐƯỢC CHỌN ---
def redraw_single_scene(scene_id, custom_kw, drive_gen_path):
    if not os.path.exists("scene_metadata.json"): return "Chưa có dữ liệu kịch bản!", None
    with open("scene_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
        
    found = False
    for s in meta:
        if s.get('sentence_id') == int(scene_id):
            found = True
            for img_idx, img in enumerate(s.get('images', [])):
                kw = custom_kw.strip() if custom_kw.strip() else img.get('svg_search_prompt', 'concept')
                target_p = os.path.join(ASSETS_DIR, f"sentence_{int(scene_id):03d}_img_{img_idx+1:02d}.svg")
                slug_n = clean_slug(kw)
                drive_p = os.path.join(drive_gen_path, f"{slug_n}_{random.randint(100,999)}.svg")
                generate_doodle_svg(kw, target_p, drive_p)
                img['svg_search_prompt'] = kw
                img['source'] = f"AI Vẽ Lại: {os.path.basename(drive_p)}"
                
    if found:
        with open("scene_metadata.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        return f"✅ Đã vẽ lại thành công ảnh cho Cảnh #{scene_id} với từ khóa '{custom_kw}'!", generate_preview_cards(meta)
    return f"❌ Không tìm thấy cảnh #{scene_id}", None

# --- GIAO DIỆN WEB GRADIO HIỆN ĐẠI ---
with gr.Blocks(title="Auto-Scribe V2 Control Center", theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate")) as app:
    gr.Markdown("# 🚀 AUTO-SCRIBE V2: TRUNG TÂM ĐIỀU KHIỂN & CẦU NỐI AI AGENT")
    gr.Markdown("Quy trình 3 bước: Bóc tách gửi Laptop ➔ Kéo kịch bản & Xem trước ảnh ➔ Đóng gói VideoScribe kèm âm thanh.")
    
    with gr.Tabs():
        with gr.TabItem("🎬 Quy Trình 3 Bước (Xem Trước & Xuất Video)"):
            with gr.Row():
                with gr.Column(scale=1):
                    audio_in = gr.File(label="🎙️ File Giọng Đọc (voiceover.mp3)", file_types=[".mp3", ".wav"])
                    bridge_url_in = gr.Textbox(label="🔗 Local Bridge URL (Cloudflare Tunnel từ Laptop)", placeholder="https://xxxx.trycloudflare.com")
                    drive_f_in = gr.Textbox(label="📂 Thư mục ảnh gốc trên Drive", value="/content/drive/MyDrive/image/f")
                    drive_gen_in = gr.Textbox(label="💾 Thư mục lưu ảnh AI sinh mới trên Drive", value="/content/drive/MyDrive/image/f/gen")
                    
                    gr.Markdown("---")
                    btn_step1 = gr.Button("📤 BƯỚC 1: Whisper Bóc Tách & Gửi Về Laptop", variant="primary")
                    btn_step2 = gr.Button("📥 BƯỚC 2: Kéo Kịch Bản & Tải/Vẽ Ảnh (Xem Trước)", variant="secondary")
                    
                    gr.Markdown("---")
                    btn_step3 = gr.Button("📦 BƯỚC 3: ĐÃ ƯNG Ý ➔ BẤM ĐÓNG GÓI & TẢI FILE SCRIBE", variant="primary", size="lg")
                
                with gr.Column(scale=1):
                    log_box = gr.Textbox(label="📋 Nhật Ký Hoạt Động (Live Console Logs)", lines=12, interactive=False)
                    out_file = gr.File(label="📦 Tải File Dự Án VideoScribe (.scribe)")
                    
            gr.Markdown("## 🖼️ Bảng Xem Trước Ảnh Từng Cảnh (Visual Preview)")
            preview_display = gr.HTML(label="Bảng Xem Trước Ảnh")
            
            with gr.Row():
                scene_select = gr.Number(label="ID Cảnh Muốn Vẽ Lại (Ví dụ: 1)", value=1, precision=0)
                custom_kw_in = gr.Textbox(label="Từ Khóa Mới Cho Cảnh Này (Tiếng Anh)", placeholder="luxury gold watch, flying rocket...")
                btn_redraw = gr.Button("🎨 AI Vẽ Lại Ảnh Cho Cảnh Này & Lưu Drive", variant="secondary")
            redraw_msg = gr.Textbox(label="Thông Báo Vẽ Lại", interactive=False)
            
    btn_step1.click(
        fn=step1_whisper_and_send_to_laptop,
        inputs=[audio_in, bridge_url_in],
        outputs=[log_box]
    )
    
    btn_step2.click(
        fn=step2_pull_from_laptop_and_preview,
        inputs=[bridge_url_in, drive_f_in, drive_gen_in],
        outputs=[log_box, preview_display]
    )
    
    btn_step3.click(
        fn=step3_build_and_download_scribe,
        inputs=[audio_in],
        outputs=[log_box, out_file]
    )
    
    btn_redraw.click(
        fn=redraw_single_scene,
        inputs=[scene_select, custom_kw_in, drive_gen_in],
        outputs=[redraw_msg, preview_display]
    )

print("🌐 Đang khởi chạy Giao Diện Web & Mở Đường Link Tunnel...")
app.queue().launch(share=True, debug=False, show_error=True)